# Random survival forest, SHAP and cross-cancer transfer (thesis_rsf kernel)

One Random Survival Forest per cohort on the expanded clinical feature set
(age, gender, overall stage, T/N/M stage, prior malignancy, residual disease,
treatment received, race, ethnicity). The forest handles censoring, so its
concordance index is comparable to Cox and its permutation importances say which
variables drive survival.

Performance is estimated with 5-fold stratified cross-validation (stratified on
the event flag). To keep the Cox comparison fair, a Cox model is scored on the
*same* folds (scikit-survival's Cox, out-of-sample). The lifelines Cox C-index
from kernel 1 (apparent / in-sample, the value in the main Cox table) is read
from Results/cox_cindex_summary.csv and shown for reference, so the figure never
hard-codes a number that could drift from the main analysis.

Why T/N/M appear here but not in the Cox table: granular staging is collinear with
overall stage, which harms Cox coefficient interpretation, so kernel 1 keeps it
out. A forest is untroubled by that collinearity, so it can exploit the extra
detail - one of the reasons to run it. Note the same correlation means stage and
T/N/M share their importance, so each looks individually smaller than staging is
collectively (discussed with the importance figure).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_curve, auc
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored, integrated_brier_score, cumulative_dynamic_auc
from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.compare import compare_survival

# shared feature definitions and helper functions, kept in one place in rsf_common.py
from rsf_common import (FEATURES, FEATURE_LABELS, COLOURS, LABELS,
                        FIGURE_DIR, RESULTS_DIR,
                        load_clean, prepare_cohort, build_xy, read_cox_cindex,
                        _complete_case, _encode_forest, _make_y)

os.makedirs(FIGURE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# forest settings shared by every section below
RSF_PARAMS   = dict(n_estimators=300, min_samples_split=10, min_samples_leaf=15,
                    max_features='sqrt', n_jobs=-1, random_state=42)
RANDOM_STATE = 42

# house colours, shared by every figure below
BLUE = '#1E6091'   # colon
TEAL = '#2A9D8F'   # rectum
RED  = '#E63946'   # stomach
NAVY = '#0D1B2A'
GREY = '#6C757D'

plt.rcParams['figure.dpi']        = 150
plt.rcParams['font.family']       = 'sans-serif'
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.titlesize']    = 13
plt.rcParams['axes.labelsize']    = 11
plt.rcParams['xtick.labelsize']   = 10
plt.rcParams['ytick.labelsize']   = 10

In [2]:
# random-forest section settings
COX_ALPHA = 0.1    # small ridge so the matched Cox stays stable on sparse folds
N_SPLITS  = 5      # cross-validation folds
N_PERM    = 20     # permutation importance repeats per fold

In [ ]:
# the clean per-patient files, read once here and reused by every section below
print("Loading clean datasets...")
datasets = load_clean()
for k, df in datasets.items():
    print(f"  {LABELS[k]}: {len(df)} patients")

cox_apparent = read_cox_cindex()   # {cohort_key: lifelines in-sample C-index} from kernel 1

## Cross-validated fit per cohort (forest + matched Cox on identical folds)

In [4]:
cindex_rows = []
importance  = {}   # cohort -> (mean, std) permutation importance per feature

for key, df in datasets.items():
    X_rsf, X_cox, y = prepare_cohort(df)
    assert (X_rsf.index == X_cox.index).all(), "forest / Cox designs are misaligned"
    event_flags = np.array([rec[0] for rec in y])

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    rsf_c, cox_c, fold_importance = [], [], []

    for train_idx, test_idx in skf.split(X_rsf, event_flags):
        y_tr, y_te = y[train_idx], y[test_idx]

        # --- Random Survival Forest ---
        rsf = RandomSurvivalForest(**RSF_PARAMS)
        rsf.fit(X_rsf.iloc[train_idx], y_tr)
        rsf_c.append(rsf.score(X_rsf.iloc[test_idx], y_te))
        perm = permutation_importance(rsf, X_rsf.iloc[test_idx], y_te,
                                      n_repeats=N_PERM, random_state=RANDOM_STATE)
        fold_importance.append(perm.importances_mean)

        # --- matched Cox on the same fold (drop dummies constant in this train split) ---
        Xc_tr, Xc_te = X_cox.iloc[train_idx], X_cox.iloc[test_idx]
        const = [c for c in Xc_tr.columns if Xc_tr[c].nunique() < 2]
        try:
            cox = CoxPHSurvivalAnalysis(alpha=COX_ALPHA)
            cox.fit(Xc_tr.drop(columns=const), y_tr)
            cox_c.append(cox.score(Xc_te.drop(columns=const), y_te))
        except Exception:
            cox_c.append(np.nan)

    rsf_c, cox_c = np.array(rsf_c), np.array(cox_c)
    fold_importance = np.array(fold_importance)
    imp_mean, imp_std = fold_importance.mean(axis=0), fold_importance.std(axis=0)
    importance[key] = (pd.Series(imp_mean, index=FEATURES), pd.Series(imp_std, index=FEATURES))

    n_events = int(event_flags.sum())
    ref = cox_apparent.get(key, np.nan)
    cindex_rows.append({
        'cohort': LABELS[key],
        'RSF_CV_mean': round(rsf_c.mean(), 3),        'RSF_CV_std': round(rsf_c.std(), 3),
        'Cox_CV_mean': round(np.nanmean(cox_c), 3),   'Cox_CV_std': round(np.nanstd(cox_c), 3),
        'Cox_apparent_lifelines': ref,
        'n': len(X_rsf), 'events': n_events,
    })

    print(f"\n{'-'*58}\n {LABELS[key]}   (n={len(X_rsf)}, events={n_events})\n{'-'*58}")
    print(f"  RSF fold C-indices : {', '.join(f'{c:.3f}' for c in rsf_c)}")
    print(f"  Cox fold C-indices : {', '.join(f'{c:.3f}' for c in cox_c)}")
    print(f"  RSF  CV C-index    : {rsf_c.mean():.3f} +/- {rsf_c.std():.3f}   (out-of-sample)")
    print(f"  Cox  CV C-index    : {np.nanmean(cox_c):.3f} +/- {np.nanstd(cox_c):.3f}   (out-of-sample, matched folds)")
    print(f"  Cox apparent       : {ref if np.isnan(ref) else f'{ref:.3f}'}   (lifelines, in-sample, from kernel 1)")
    print("  Permutation importance (mean +/- sd across folds):")
    for i in imp_mean.argsort()[::-1]:
        print(f"    {FEATURE_LABELS[FEATURES[i]]:<20} {imp_mean[i]:+.4f} +/- {imp_std[i]:.4f}")

    out = pd.DataFrame({'feature': [FEATURE_LABELS[f] for f in FEATURES],
                        'importance_mean': imp_mean, 'importance_std': imp_std}
                       ).sort_values('importance_mean', ascending=False)
    out.to_csv(f'{RESULTS_DIR}/rsf_importance_{key}.csv', index=False)
    print(f"  Saved: {RESULTS_DIR}/rsf_importance_{key}.csv")

cindex_df = pd.DataFrame(cindex_rows)
cindex_df.to_csv(f'{RESULTS_DIR}/rsf_cindex_summary.csv', index=False)
print(f"\nSaved: {RESULTS_DIR}/rsf_cindex_summary.csv")


----------------------------------------------------------
 COAD (Colon)   (n=426, events=90)
----------------------------------------------------------
  RSF fold C-indices : 0.809, 0.737, 0.828, 0.674, 0.739
  Cox fold C-indices : 0.756, 0.695, 0.821, 0.602, 0.760
  RSF  CV C-index    : 0.757 +/- 0.055   (out-of-sample)
  Cox  CV C-index    : 0.727 +/- 0.074   (out-of-sample, matched folds)
  Cox apparent       : 0.762   (lifelines, in-sample, from kernel 1)
  Permutation importance (mean +/- sd across folds):
    Overall stage        +0.0418 +/- 0.0095
    N stage              +0.0346 +/- 0.0336
    Age                  +0.0279 +/- 0.0231
    M stage              +0.0206 +/- 0.0096
    T stage              +0.0134 +/- 0.0062
    Treatment received   +0.0069 +/- 0.0076
    Residual disease     +0.0043 +/- 0.0119
    Prior malignancy     +0.0018 +/- 0.0026
    Gender (Male)        +0.0003 +/- 0.0034
    Ethnicity            -0.0005 +/- 0.0052
    Race                 -0.0060 +/- 0.00

## Figure 13: feature importance (mean +/- sd across folds)
Stage and T/N/M are correlated, so their importance is shared out between them -
read staging as a group rather than judging each bar alone.

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(17, 6), sharey=True)
fig.suptitle('Figure 13: Random Survival Forest Feature Importance (5-fold CV)',
             fontsize=14, fontweight='bold', y=1.02)
for ax, (key, (imp_mean, imp_std)) in zip(axes, importance.items()):
    order = imp_mean.reindex(FEATURES).sort_values().index
    m, s = imp_mean.reindex(order), imp_std.reindex(order)
    ax.barh([FEATURE_LABELS[f] for f in order], m.values, xerr=s.values,
            color=COLOURS[key], edgecolor='white', capsize=3)
    ax.axvline(0, color='grey', linewidth=0.8)
    ax.set_title(LABELS[key]); ax.set_xlabel('Mean drop in C-index (permutation)')
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig13_rsf_importance.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: fig13_rsf_importance.png")

Saved: fig13_rsf_importance.png


## Figure 14: predictive performance, Cox vs Random Survival Forest
The two cross-validated bars (Cox CV and RSF CV) are the fair comparison: both
scored out-of-sample on the same folds. The apparent Cox bar (in-sample, from the
main analysis) is shown so the optimism of the in-sample number is visible.

In [6]:
keys = list(datasets.keys())
sidx = cindex_df.set_index('cohort')
have_ref = not cindex_df['Cox_apparent_lifelines'].isna().all()

cox_app = [sidx.loc[LABELS[k], 'Cox_apparent_lifelines'] for k in keys]
cox_cv  = [sidx.loc[LABELS[k], 'Cox_CV_mean'] for k in keys]
cox_cvs = [sidx.loc[LABELS[k], 'Cox_CV_std']  for k in keys]
rsf_cv  = [sidx.loc[LABELS[k], 'RSF_CV_mean'] for k in keys]
rsf_cvs = [sidx.loc[LABELS[k], 'RSF_CV_std']  for k in keys]

x = np.arange(len(keys))
w = 0.27 if have_ref else 0.38
fig, ax = plt.subplots(figsize=(10, 5.5))
if have_ref:
    ax.bar(x - w, cox_app, w, label='Cox (apparent, in-sample)', color='#9aa5b1', edgecolor='white')
    ax.bar(x,     cox_cv,  w, yerr=cox_cvs, capsize=4, label='Cox (5-fold CV)', color='#0D1B2A', edgecolor='white')
    ax.bar(x + w, rsf_cv,  w, yerr=rsf_cvs, capsize=4, label='RSF (5-fold CV)', color='#1E6091', edgecolor='white')
else:
    ax.bar(x - w/2, cox_cv, w, yerr=cox_cvs, capsize=4, label='Cox (5-fold CV)', color='#0D1B2A', edgecolor='white')
    ax.bar(x + w/2, rsf_cv, w, yerr=rsf_cvs, capsize=4, label='RSF (5-fold CV)', color='#1E6091', edgecolor='white')
ax.axhline(0.5, color='grey', linestyle='--', linewidth=1, label='No discrimination (0.5)')
ax.set_xticks(x); ax.set_xticklabels([LABELS[k] for k in keys])
ax.set_ylabel('Concordance index (C-index)'); ax.set_ylim(0.4, 1.0)
ax.set_title('Figure 14: Predictive Performance, Cox vs Random Survival Forest', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig14_cox_vs_rsf_cindex.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: fig14_cox_vs_rsf_cindex.png")

print(f"\nRandom Survival Forest complete - figures in {FIGURE_DIR}, tables in {RESULTS_DIR}")

Saved: fig14_cox_vs_rsf_cindex.png

Random Survival Forest complete - figures in Figures, tables in Results


## SHAP feature importance

Explains the forest with SHAP, on the same expanded clinical feature set and the
same encoding as random_survival_forest.py (both import rsf_common, so the
explanation matches the model). Two views per cohort:
  - Figure 15: global importance, mean |SHAP| per feature
  - Figure 16: a beeswarm, one dot per patient coloured by feature value, showing
    how each feature pushes that patient's predicted risk up or down

What is explained: the forest's risk score (higher = higher risk / shorter
survival). A positive SHAP value pushes a patient towards higher risk.

Method note: SHAP's fast TreeExplainer does not support scikit-survival's forest,
so this uses the model-agnostic PERMUTATION explainer on the risk score. The exact
explainer enumerates every feature coalition (2^features), which is fine for three
features but intractable for eleven; the permutation explainer scales.

In [7]:
# SHAP section settings
BG_SIZE      = 50     # background sample the permutation masker draws "off" values from
EXPLAIN_SIZE = 150    # patients explained per cohort (mean |SHAP| is stable well below full n; keeps runtime sane)
MAX_EVALS    = 200    # permutation explainer evaluations per patient (>= 2*n_features+1)

DISPLAY_NAMES = [FEATURE_LABELS[f] for f in FEATURES]

## Compute SHAP values per cohort and draw the beeswarm (Figure 16)

In [9]:
mean_abs_shap = {}   # cohort -> Series of mean|SHAP| per feature

for key, df in datasets.items():
    X, y = build_xy(df)                      # same 11-feature encoding as the forest
    rsf = RandomSurvivalForest(**RSF_PARAMS).fit(X, y)

    # sanity check: the risk score must actually vary across patients
    rng = float(np.ptp(rsf.predict(X)))
    print(f"\n{LABELS[key]}  (n={len(X)})")
    print(f"  [check] risk-score range across patients: {rng:.3f}  (should be > 0)")

    # SHAP hands the function either a numpy array OR a labelled frame; rebuild the
    # frame BY POSITION (np.asarray drops labels) so columns line up with the model.
    def predict_risk(data):
        arr = np.asarray(data, dtype=float)
        return rsf.predict(pd.DataFrame(arr, columns=FEATURES))

    X_display  = X.rename(columns=FEATURE_LABELS)                 # nice names on the plots
    background = X_display.sample(min(BG_SIZE, len(X_display)), random_state=RANDOM_STATE)
    explain_on = X_display.sample(min(EXPLAIN_SIZE, len(X_display)), random_state=RANDOM_STATE)

    explainer   = shap.Explainer(predict_risk, background, algorithm='permutation')
    shap_values = explainer(explain_on, max_evals=MAX_EVALS)

    mean_abs = np.abs(shap_values.values).mean(axis=0)
    mean_abs_shap[key] = pd.Series(mean_abs, index=DISPLAY_NAMES)

    print("  mean |SHAP| per feature:")
    for name, val in mean_abs_shap[key].sort_values(ascending=False).items():
        print(f"    {name:<20} {val:.4f}")

    pd.DataFrame({'feature': DISPLAY_NAMES, 'mean_abs_shap': mean_abs}) \
        .sort_values('mean_abs_shap', ascending=False) \
        .to_csv(f'{RESULTS_DIR}/shap_importance_{key}.csv', index=False)

    # beeswarm (Figure 16)
    shap.plots.beeswarm(shap_values, max_display=len(FEATURES), show=False, color_bar=True)
    fig = plt.gcf(); fig.set_size_inches(8, 5)
    plt.title(f'SHAP summary: {LABELS[key]}', fontsize=13, fontweight='bold')
    plt.xlabel('SHAP value (impact on predicted risk)')
    plt.tight_layout()
    plt.savefig(f'{FIGURE_DIR}/fig16_shap_beeswarm_{key}.png', bbox_inches='tight', dpi=150)
    plt.close('all')
    print(f"  Saved: fig16_shap_beeswarm_{key}.png")


COAD (Colon)  (n=426)
  [check] risk-score range across patients: 47.134  (should be > 0)


PermutationExplainer explainer: 151it [04:03,  1.64s/it]                         


  mean |SHAP| per feature:
    Overall stage        3.2920
    N stage              2.8306
    Age                  2.5315
    M stage              2.0996
    Residual disease     1.3055
    T stage              0.9341
    Treatment received   0.5894
    Prior malignancy     0.2596
    Race                 0.2007
    Ethnicity            0.1238
    Gender (Male)        0.1225
  Saved: fig16_shap_beeswarm_coad.png

READ (Rectum)  (n=153)
  [check] risk-score range across patients: 8.689  (should be > 0)


PermutationExplainer explainer: 151it [02:34,  1.10s/it]                         


  mean |SHAP| per feature:
    Age                  1.0627
    Overall stage        0.7973
    N stage              0.6154
    M stage              0.4939
    Treatment received   0.3763
    T stage              0.1042
    Race                 0.0741
    Gender (Male)        0.0486
    Ethnicity            0.0342
    Residual disease     0.0096
    Prior malignancy     0.0000
  Saved: fig16_shap_beeswarm_read.png

STAD (Stomach)  (n=393)
  [check] risk-score range across patients: 83.341  (should be > 0)


PermutationExplainer explainer: 151it [05:05,  2.11s/it]                         


  mean |SHAP| per feature:
    Overall stage        6.6236
    Treatment received   6.2146
    N stage              5.1747
    Residual disease     4.4005
    Age                  4.2054
    T stage              1.8895
    Race                 0.5553
    Gender (Male)        0.4790
    Ethnicity            0.3938
    M stage              0.3178
    Prior malignancy     0.0000
  Saved: fig16_shap_beeswarm_stad.png


## Figure 15: global SHAP importance (mean |SHAP value|), house style

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(17, 6), sharey=True)
fig.suptitle('Figure 15: Global SHAP Importance (mean |SHAP value|)',
             fontsize=14, fontweight='bold', y=1.02)
for ax, (key, s) in zip(axes, mean_abs_shap.items()):
    s = s.sort_values()
    ax.barh(s.index, s.values, color=COLOURS[key], edgecolor='white')
    ax.set_title(LABELS[key]); ax.set_xlabel('mean |SHAP value|')
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig15_shap_importance.png', bbox_inches='tight', dpi=150)
plt.close()
print("\nSaved: fig15_shap_importance.png")

print(f"\nSHAP complete - figures in {FIGURE_DIR}, tables in {RESULTS_DIR}")


Saved: fig15_shap_importance.png

SHAP complete - figures in Figures, tables in Results


## Transfer survival: train on colorectal, test on stomach

Supervisor question 2: what can colorectal survival patterns tell us about stomach
cancer? This fits a random survival forest on the pooled colorectal cohorts (COAD +
READ) and evaluates it, without any refitting, on the stomach cohort (STAD). If a
colorectal-trained risk model can still rank and calibrate stomach patients, the two
cancers share prognostic structure.

This is the direct external-validation baseline. It is also the baseline that the
transfer survival forest (TSF) method is designed to improve on, so it is needed
either way. Metrics: concordance index, integrated Brier score, and time-dependent
AUC, exactly as requested. It also splits the stomach patients by their
colorectal-predicted risk and compares their survival, which is the visual form of
"high colorectal risk maps to high stomach risk".

Everything uses scikit-survival, so no lifelines is needed; the Kaplan-Meier and
log-rank pieces use sksurv's own estimators. Feature encoding is imported from
rsf_common, so it is identical to the main forest.

In [11]:
# transfer section: within-stomach CV C-index from the forest section, shown for reference
STAD_CV_CINDEX = 0.650

In [ ]:
# ---- build source (colorectal = COAD + READ) and target (stomach = STAD) ----
crc_raw = pd.concat([datasets['coad'], datasets['read']], ignore_index=True)
crc  = _complete_case(crc_raw)          # complete-case on age + stage, same rule as the forest
stad = _complete_case(datasets['stad'])

X_train, y_train = _encode_forest(crc),  _make_y(crc)     # source: colorectal
X_test,  y_test  = _encode_forest(stad), _make_y(stad)    # target: stomach
# _encode_forest enforces the same FEATURES order, so train and test columns align

ev_tr = np.array([e for e, t in y_train]); tm_tr = np.array([t for e, t in y_train])
ev_te = np.array([e for e, t in y_test]);  tm_te = np.array([t for e, t in y_test])

print("Transfer setup")
print(f"  Source (colorectal, COAD+READ): n={len(X_train)}, deaths={int(ev_tr.sum())}")
print(f"  Target (stomach, STAD):         n={len(X_test)},  deaths={int(ev_te.sum())}")

In [13]:
# ---- fit on colorectal, predict on stomach ----
rsf = RandomSurvivalForest(**RSF_PARAMS).fit(X_train, y_train)
risk = rsf.predict(X_test)              # higher = higher predicted risk

## Metric 1: concordance index on the stomach cohort

In [14]:
cindex = concordance_index_censored(ev_te.astype(bool), tm_te, risk)[0]
print(f"\nC-index (colorectal model, evaluated on stomach): {cindex:.3f}")
print(f"  For reference, a stomach-trained forest scores {STAD_CV_CINDEX:.3f} under 5-fold CV.")
print(f"  Closeness to that value indicates how well colorectal risk structure transfers;")
print(f"  0.5 would mean no better than chance.")


C-index (colorectal model, evaluated on stomach): 0.646
  For reference, a stomach-trained forest scores 0.650 under 5-fold CV.
  Closeness to that value indicates how well colorectal risk structure transfers;
  0.5 would mean no better than chance.


## Time grid for the time-dependent metrics
The Brier score and time-dependent AUC are only defined where the inverse-probability
-of-censoring weights are estimable, which requires the evaluation times to lie inside
the observed event-time range of BOTH cohorts. The grid is therefore clipped to that
overlap; this is the step that otherwise throws an out-of-range error.

In [15]:
et_tr = tm_tr[ev_tr.astype(bool)]      # event times, source
et_te = tm_te[ev_te.astype(bool)]      # event times, target
lo = max(et_tr.min(), et_te.min())
hi = min(et_tr.max(), et_te.max())
eps = 1e-3 * (hi - lo)
grid = np.unique(np.percentile(et_te, np.linspace(10, 90, 15)))
times = grid[(grid > lo + eps) & (grid < hi - eps)]
if len(times) < 5:                     # fallback: even grid strictly inside the overlap
    times = np.linspace(lo + eps, hi - eps, 12)
print(f"\nEvaluation window: {times.min():.0f} to {times.max():.0f} days ({len(times)} points)")


Evaluation window: 95 to 793 days (15 points)


## Metric 2: integrated Brier score

In [16]:
surv_fns = rsf.predict_survival_function(X_test)
surv_prob = np.row_stack([fn(times) for fn in surv_fns])        # (n_stomach, n_times)
ibs = integrated_brier_score(y_train, y_test, surv_prob, times)
print(f"Integrated Brier score on stomach: {ibs:.3f}   (lower is better; 0.25 is uninformative)")

Integrated Brier score on stomach: 0.192   (lower is better; 0.25 is uninformative)


C:\Users\anmol\AppData\Local\Temp\ipykernel_3372\3530409630.py:2: DeprecationWarning: `row_stack` alias is deprecated. Use `np.vstack` directly.
  surv_prob = np.row_stack([fn(times) for fn in surv_fns])        # (n_stomach, n_times)


## Metric 3: time-dependent AUC

In [17]:
auc_t, mean_auc = cumulative_dynamic_auc(y_train, y_test, risk, times)
print(f"Mean time-dependent AUC on stomach: {mean_auc:.3f}   (0.5 is chance)")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(times, auc_t, marker='o', color=COLOURS['stad'], linewidth=2)
ax.axhline(mean_auc, color='grey', linestyle='--', linewidth=1, label=f'Mean AUC = {mean_auc:.3f}')
ax.axhline(0.5, color='black', linestyle=':', linewidth=1, label='Chance (0.5)')
ax.set_xlabel('Time (days)'); ax.set_ylabel('Time-dependent AUC')
ax.set_ylim(0.4, 1.0)
ax.set_title('Figure 14 (transfer): Colorectal model, time-dependent AUC on stomach', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig_transfer_auc.png', bbox_inches='tight', dpi=150)
plt.close()
print(f"  Saved: {FIGURE_DIR}/fig_transfer_auc.png")

Mean time-dependent AUC on stomach: 0.671   (0.5 is chance)
  Saved: Figures/fig_transfer_auc.png


## Does high colorectal-predicted risk mean poorer stomach survival?
The stomach patients are split into three groups by the colorectal model's risk score
and their Kaplan-Meier curves compared. A clean separation, low-risk surviving best,
is the visual confirmation that colorectal risk transfers to stomach.

In [18]:
tertiles = np.quantile(risk, [1/3, 2/3])
group = np.digitize(risk, tertiles)                # 0 = low, 1 = medium, 2 = high (colorectal-predicted risk)
labels = {0: 'Low CRC-predicted risk', 1: 'Medium', 2: 'High CRC-predicted risk'}
band_colours = {0: '#2A9D8F', 1: '#E9A800', 2: '#E63946'}

chi2, pval = compare_survival(y_test, group)       # log-rank across the three risk groups
print(f"\nStomach survival by colorectal-predicted risk group: log-rank p = {pval:.3g}")

fig, ax = plt.subplots(figsize=(8.5, 5.5))
for g in (0, 1, 2):
    mask = group == g
    t, s = kaplan_meier_estimator(ev_te[mask].astype(bool), tm_te[mask])
    ax.step(t, s, where='post', color=band_colours[g], linewidth=2,
            label=f'{labels[g]} (n={int(mask.sum())}, d={int(ev_te[mask].sum())})')
ax.set_xlabel('Time (days)'); ax.set_ylabel('Survival probability'); ax.set_ylim(0, 1.02)
ax.set_title(f'Figure 15 (transfer): Stomach survival by colorectal-predicted risk\nlog-rank p = {pval:.3g}',
             fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig_transfer_km_risk.png', bbox_inches='tight', dpi=150)
plt.close()
print(f"  Saved: {FIGURE_DIR}/fig_transfer_km_risk.png")


Stomach survival by colorectal-predicted risk group: log-rank p = 1.54e-07
  Saved: Figures/fig_transfer_km_risk.png


In [19]:
# ---- save the metrics ----
pd.DataFrame([{
    'direction': 'train colorectal (COAD+READ), test stomach (STAD)',
    'n_train': len(X_train), 'deaths_train': int(ev_tr.sum()),
    'n_test': len(X_test),  'deaths_test': int(ev_te.sum()),
    'C_index': round(cindex, 3),
    'stomach_internal_CV_C_index': STAD_CV_CINDEX,
    'integrated_brier_score': round(ibs, 3),
    'mean_time_dependent_AUC': round(mean_auc, 3),
    'risk_group_logrank_p': float(f'{pval:.3g}'),
}]).to_csv(f'{RESULTS_DIR}/transfer_metrics.csv', index=False)
print(f"\nSaved: {RESULTS_DIR}/transfer_metrics.csv")
print("\nTransfer baseline complete.")


Saved: Results/transfer_metrics.csv

Transfer baseline complete.


## Metric 4: ROC-AUC at one, two and three years

A plain ROC curve is not valid here. Patients lost to follow-up before the horizon have an unknown
status, so they cannot simply be labelled alive or dead. The standard fix is a time-dependent ROC
with inverse-probability-of-censoring weights: each patient is weighted by the inverse of their
probability of still being under observation.

    cases    = died on or before the horizon    weight 1 / G(T_i)
    controls = still alive at the horizon       weight 1 / G(horizon)
    censored before the horizon                 dropped, status unknown

G is the Kaplan-Meier estimate of the censoring distribution. This is the survival equivalent of a
classification ROC curve, and it should agree with the mean time-dependent AUC from Metric 3.

In [ ]:
HORIZONS = [365, 730, 1095]      # one, two and three years


def _censoring_km(ev, tm):
    """Kaplan-Meier estimate of the censoring distribution G(t): the event flag reversed."""
    return kaplan_meier_estimator(~ev.astype(bool), tm)


def _G(grid_t, grid_s, q):
    i = np.searchsorted(grid_t, q, side='right') - 1
    return grid_s[i] if i >= 0 else 1.0


def td_roc(ev, tm, score, horizon):
    """Time-dependent ROC at a fixed horizon, censoring-weighted.
    Returns the false-positive rate, true-positive rate, AUC and the case/control counts."""
    ev = ev.astype(bool)
    gt, gs = _censoring_km(ev, tm)
    lab, wt, sc = [], [], []
    for i in range(len(tm)):
        if tm[i] <= horizon and ev[i]:               # case: died by the horizon
            g = _G(gt, gs, tm[i])
            if g > 0:
                lab.append(1); wt.append(1 / g); sc.append(score[i])
        elif tm[i] > horizon:                        # control: still alive at the horizon
            g = _G(gt, gs, horizon)
            if g > 0:
                lab.append(0); wt.append(1 / g); sc.append(score[i])
        # anyone censored before the horizon is dropped: their status is unknown
    lab, wt, sc = np.array(lab), np.array(wt), np.array(sc)
    fpr, tpr, _ = roc_curve(lab, sc, sample_weight=wt)
    order = np.argsort(fpr)                          # a weighted ROC can come back unsorted
    fpr, tpr = fpr[order], tpr[order]
    return fpr, tpr, auc(fpr, tpr), int((lab == 1).sum()), int((lab == 0).sum())


def oof_risk(X, y, ev, make_model):
    """Out-of-fold risk: every patient scored by a model that never saw them. This is the honest
    within-cohort ceiling to judge a transferred model against."""
    out = np.zeros(len(X))
    folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for tr, te in folds.split(X, ev):
        out[te] = make_model().fit(X.iloc[tr], y[tr]).predict(X.iloc[te])
    return out

In [ ]:
# ---- ROC of the transferred forest on stomach, at each horizon ----
# `risk` is already the colorectal forest applied to every stomach patient (computed above).
roc_rows = []
fig, ax = plt.subplots(figsize=(6.5, 6))
for h, colour in zip(HORIZONS, [BLUE, TEAL, RED]):
    fpr, tpr, a, n_case, n_ctrl = td_roc(ev_te, tm_te, risk, h)
    ax.plot(fpr, tpr, color=colour, lw=2, label=f'{h // 365} year  (AUC = {a:.3f})')
    roc_rows.append({'model': 'Colorectal forest -> stomach (transfer)', 'horizon_days': h,
                     'auc': round(a, 3), 'cases': n_case, 'controls': n_ctrl})
    print(f"  ROC at {h:>4} days: AUC {a:.3f}   ({n_case} cases, {n_ctrl} controls)")

ax.plot([0, 1], [0, 1], ls=':', color=GREY, lw=1.2, label='Chance (AUC = 0.500)')
ax.set_xlabel('False positive rate (1 - specificity)')
ax.set_ylabel('True positive rate (sensitivity)')
ax.set_title('Transfer learning ROC on stomach\n'
             'colorectal-trained forest, never shown a stomach patient', fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig_transfer_roc.png', bbox_inches='tight', dpi=300)
plt.close()
print("\nSaved: fig_transfer_roc.png")
print(f"  Metric 3 gave a mean time-dependent AUC of {mean_auc:.3f}, so the two agree.")

## Comparing methods and directions of transfer

Three things the C-index on its own does not settle:

1. Does the **method** matter? Swap the forest for a Cox model and transfer that instead.
2. How close does transfer get to a model trained on the target cohort itself? That ceiling is
   estimated out of fold, so every patient is scored by a model that never saw them.
3. Does transfer also work in the **other direction**, stomach to colorectal?

The C-index is only comparable *within* a target cohort, never between cohorts, because it depends
on how hard that cohort is to rank. Each direction is therefore judged against its own ceiling, not
against the other direction.

In [ ]:
# ---- the comparison models ----
# The colorectal forest (rsf), its stomach predictions (risk) and its C-index (cindex) already exist
# from the section above, so only the genuinely new models are fitted here.
print("Fitting the comparison models, this takes a few minutes...")

# same transfer, different method: a Cox model in place of the forest
risk_cox = CoxPHSurvivalAnalysis(alpha=COX_ALPHA).fit(X_train, y_train).predict(X_test)

# the within-stomach ceiling: every stomach patient scored out of fold
risk_stad_oof = oof_risk(X_test, y_test, ev_te, lambda: RandomSurvivalForest(**RSF_PARAMS))

# the reverse direction: a stomach-trained forest applied to colorectal
risk_reverse = RandomSurvivalForest(**RSF_PARAMS).fit(X_test, y_test).predict(X_train)

# and the within-colorectal ceiling, to judge that reverse transfer against
risk_crc_oof = oof_risk(X_train, y_train, ev_tr, lambda: RandomSurvivalForest(**RSF_PARAMS))

c_cox      = concordance_index_censored(ev_te.astype(bool), tm_te, risk_cox)[0]
c_stad_oof = concordance_index_censored(ev_te.astype(bool), tm_te, risk_stad_oof)[0]
c_reverse  = concordance_index_censored(ev_tr.astype(bool), tm_tr, risk_reverse)[0]
c_crc_oof  = concordance_index_censored(ev_tr.astype(bool), tm_tr, risk_crc_oof)[0]

print(f"\nTarget = stomach    (n={len(X_test)}, deaths={int(ev_te.sum())})")
print(f"  colorectal forest -> stomach (transfer)  : {cindex:.3f}")
print(f"  colorectal Cox    -> stomach (transfer)  : {c_cox:.3f}")
print(f"  stomach forest, out of fold (ceiling)    : {c_stad_oof:.3f}")
print(f"  transfer reaches {100 * cindex / c_stad_oof:.0f}% of the ceiling")

print(f"\nTarget = colorectal (n={len(X_train)}, deaths={int(ev_tr.sum())})")
print(f"  stomach forest -> colorectal (transfer)  : {c_reverse:.3f}")
print(f"  colorectal forest, out of fold (ceiling) : {c_crc_oof:.3f}")
print(f"  transfer reaches {100 * c_reverse / c_crc_oof:.0f}% of the ceiling")

In [ ]:
# ---- two-year ROC, three models: does the method or the training cohort change anything? ----
H = 730                                     # two years
fig, ax = plt.subplots(figsize=(6.5, 6))
for score, name, colour, style in [
        (risk,          'Colorectal forest -> stomach (transfer)', BLUE, '-'),
        (risk_cox,      'Colorectal Cox -> stomach (transfer)',    TEAL, '-'),
        (risk_stad_oof, 'Stomach forest, out-of-fold (ceiling)',   RED,  '--')]:
    fpr, tpr, a, n_case, n_ctrl = td_roc(ev_te, tm_te, score, H)
    ax.plot(fpr, tpr, color=colour, lw=2, ls=style, label=f'{name}\n     AUC = {a:.3f}')
    roc_rows.append({'model': name, 'horizon_days': H, 'auc': round(a, 3),
                     'cases': n_case, 'controls': n_ctrl})

ax.plot([0, 1], [0, 1], ls=':', color=GREY, lw=1.2, label='Chance (AUC = 0.500)')
ax.set_xlabel('False positive rate (1 - specificity)')
ax.set_ylabel('True positive rate (sensitivity)')
ax.set_title('Two-year ROC on stomach: transfer vs within-cohort', fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig_transfer_method_roc.png', bbox_inches='tight', dpi=300)
plt.close()
print("Saved: fig_transfer_method_roc.png")

In [ ]:
# ---- each transfer against its own target's ceiling, in both directions ----
# One panel per target cohort: a C-index measured on stomach cannot be compared with one
# measured on colorectal, so the two panels are read separately.
fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
top = max(cindex, c_cox, c_stad_oof, c_reverse, c_crc_oof) + 0.06

panels = [
    (axes[0], f'Target: stomach  (n={len(X_test)})',
     ['Colorectal forest\n(transfer)', 'Colorectal Cox\n(transfer)', 'Stomach forest\n(ceiling)'],
     [cindex, c_cox, c_stad_oof], [BLUE, TEAL, RED], c_stad_oof),
    (axes[1], f'Target: colorectal  (n={len(X_train)})',
     ['Stomach forest\n(transfer)', 'Colorectal forest\n(ceiling)'],
     [c_reverse, c_crc_oof], [BLUE, RED], c_crc_oof),
]
for ax, title, labels, vals, colours, ceiling in panels:
    bars = ax.bar(labels, vals, color=colours, width=0.6)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.006, f'{v:.3f}',
                ha='center', fontweight='bold', color=NAVY, fontsize=11)
    ax.axhline(ceiling, ls='--', color=GREY, lw=1.1)      # this target's own ceiling
    ax.axhline(0.5, ls=':', color=GREY, lw=1.1)           # chance
    ax.set_ylim(0.45, top)
    ax.set_title(title, fontweight='bold', color=NAVY)
axes[0].set_ylabel('Concordance index')
fig.text(0.5, -0.04,
         "Dashed line marks each target's within-cohort ceiling. The C-index is only comparable within a "
         "target cohort,\nnot between them, because colorectal is intrinsically easier to rank than stomach.",
         ha='center', fontsize=9, color=GREY, style='italic')
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig_transfer_cindex_compare.png', bbox_inches='tight', dpi=300)
plt.close()
print("Saved: fig_transfer_cindex_compare.png")

In [ ]:
# ---- save the comparison numbers ----
pd.DataFrame(roc_rows).to_csv(f'{RESULTS_DIR}/transfer_roc_auc.csv', index=False)

pd.DataFrame([
    {'target': 'Stomach',    'model': 'Colorectal forest (transfer)',             'c_index': round(cindex, 3)},
    {'target': 'Stomach',    'model': 'Colorectal Cox (transfer)',                'c_index': round(c_cox, 3)},
    {'target': 'Stomach',    'model': 'Stomach forest, out-of-fold (ceiling)',    'c_index': round(c_stad_oof, 3)},
    {'target': 'Colorectal', 'model': 'Stomach forest (transfer)',                'c_index': round(c_reverse, 3)},
    {'target': 'Colorectal', 'model': 'Colorectal forest, out-of-fold (ceiling)', 'c_index': round(c_crc_oof, 3)},
]).to_csv(f'{RESULTS_DIR}/transfer_comparison.csv', index=False)

print(f"Saved: transfer_roc_auc.csv and transfer_comparison.csv in {RESULTS_DIR}")

## Data adapter for the transfer survival forest (TSF) repo

Prepares your cohorts for Yonghao Zhao's Transfer Survival Forest repo
(github.com/YonghaoZhao722/TSF), for supervisor question 2, the full transfer method.
TSF trains a source forest on a large cohort and fine-tunes a target forest on a
smaller one, so here colorectal (COAD + READ) is the SOURCE and stomach (STAD) is the
TARGET, matching the train-on-colorectal, test-on-stomach direction.

The repo's documented layout is one features CSV and one labels CSV per dataset, in
preprocessed_data/, with the source and target sharing the same feature names. This
script writes exactly that, reusing the same 11-feature encoding as the main forest
(via rsf_common), so the two cohorts are aligned by construction.

IMPORTANT, PLEASE VERIFY ONE THING. I could not read the repo's code remotely to
confirm the exact column headers in the labels CSV. Open the repo's sample labels
file (in preprocessed_data/ after their preprocess_data.py runs, or the example in the
repo) or model/methods.py, check the two column names it uses for the event flag and
the survival time, and set EVENT_COL and TIME_COL below to match. Everything else
follows the documented structure. If their sample uses a structured array saved with,
for instance, 'status' and 'survival_time', the defaults below are already correct;
if it uses 'event' and 'time', change the two constants.

After running this, copy the four files in preprocessed_data/ into TSF/preprocessed_data/.

In [ ]:
# ---- settings to check against the repo ----
SOURCE_NAME = 'SEER'   # the repo's source name (train_source_forest.py loads this); colorectal goes here
TARGET_NAME = 'wch'    # the repo's target name (target_forest_finetune.py loads this); stomach goes here
EVENT_COL   = 'status'          # <-- VERIFY against a sample labels CSV; event flag (1 = death, 0 = censored)
TIME_COL    = 'survival_time'   # <-- VERIFY against a sample labels CSV; survival time
OUT_DIR     = 'preprocessed_data'
# Survival time is left in DAYS (our native unit). Source and target use the same unit,
# which is what matters. To match a months-based repo sample, divide by 30.44.

os.makedirs(OUT_DIR, exist_ok=True)


def export(df_list, name, tag):
    d = _complete_case(pd.concat(df_list, ignore_index=True))
    X = _encode_forest(d)                                   # 11 aligned numeric features, FEATURES order
    y = pd.DataFrame({
        EVENT_COL: (d['event'].astype(int)).values,        # 1 = death, 0 = censored
        TIME_COL:  (d['survival_time'].astype(float)).values,
    })
    X.to_csv(f'{OUT_DIR}/{name}_X.csv', index=False)
    y.to_csv(f'{OUT_DIR}/{name}_y.csv', index=False)
    print(f"  {tag}: {name}_X.csv ({X.shape[0]} rows x {X.shape[1]} features), "
          f"{name}_y.csv (deaths {int(y[EVENT_COL].sum())}/{len(y)})")
    return X, y


print("Exporting for TSF (source = colorectal, target = stomach)")
Xs, ys = export([datasets['coad'], datasets['read']], SOURCE_NAME, 'SOURCE colorectal (COAD+READ)')
Xt, yt = export([datasets['stad']],                 TARGET_NAME, 'TARGET stomach (STAD)')

# ---- alignment check (TSF requires identical feature names) ----
assert list(Xs.columns) == list(Xt.columns) == FEATURES, "feature columns are not aligned"
print(f"\nFeature alignment OK: {len(FEATURES)} shared features -> {FEATURES}")
print("\nLabels CSV format written as:")
print(pd.concat([ys.head(3)], axis=0).to_string(index=False))
print(f"\nSaved four files to {OUT_DIR}/. Copy them into TSF/preprocessed_data/.")
print("If train_source_forest.py / target_forest_finetune.py use different dataset")
print("names than SEER / wch, either rename these files or edit global_names.py to match.")

## Transfer learning curve

The honest analogue of the "AUC vs training step" example. A random survival forest
has no training steps, so instead we vary the amount of STOMACH training data and, at
each size, compare:
  (a) stomach only                      - a forest trained on that many stomach patients
  (b) colorectal + stomach (transfer)   - the same stomach patients pooled with all of
                                          colorectal (COAD + READ), i.e. transfer by
                                          source-data augmentation
both scored on the SAME held-out stomach test set. A horizontal reference shows pure
colorectal transfer (no stomach training at all, the kernel-2 baseline). If transfer
helps, (b) sits above (a), most when stomach data is scarce, then they converge. Same
story as the example, told correctly for a forest.

Reuses load_clean / _complete_case / _encode_forest / _make_y and the RSF settings from
rsf_common, so the cohorts and model match transfer_survival.py exactly.

In [ ]:
# learning-curve section settings
TEST_FRAC = 0.35    # fixed stomach hold-out for scoring
N_REP     = 15      # training-subsample repeats per size, averaged (lower this to go faster)

In [ ]:
# ---- the source and target sets already exist from the transfer section above:
#      X_train / y_train = colorectal (the source), X_test / y_test = stomach (the target) ----

# A fixed stomach hold-out to score against; the stomach TRAIN pool is grown from the rest.
# Called X_hold / y_hold, not X_test / y_test, so it cannot be confused with the full stomach cohort.
X_pool, X_hold, y_pool, y_hold = train_test_split(
    X_test, y_test, test_size=TEST_FRAC, random_state=RANDOM_STATE, stratify=ev_te)
X_pool = X_pool.reset_index(drop=True)
pool_n = len(X_pool)
print(f"Source colorectal: n={len(X_train)}, deaths={int(ev_tr.sum())}")
print(f"Target stomach:    n={len(X_test)}, deaths={int(ev_te.sum())}")
print(f"Stomach train pool: {pool_n}   |   stomach hold-out: {len(X_hold)}")

sizes = sorted(set([s for s in [30, 60, 90, 120, 160, 200] if s < pool_n] + [pool_n]))

In [ ]:
# ---- pure colorectal transfer (no stomach training), scored on the same stomach hold-out ----
# `rsf` is already that model: the forest fitted on the full colorectal cohort further up.
pure_transfer = rsf.score(X_hold, y_hold)
print(f"\nPure colorectal transfer (0 stomach patients): C-index {pure_transfer:.3f}")

# ---- learning curve ----
stomach_only, with_transfer = [], []
for n in sizes:
    a, b = [], []
    for rep in range(N_REP):
        rng = np.random.RandomState(rep)
        idx = rng.choice(pool_n, size=min(n, pool_n), replace=False)
        Xs, ys = X_pool.iloc[idx], y_pool[idx]

        # (a) stomach only
        m_a = RandomSurvivalForest(**RSF_PARAMS).fit(Xs, ys)
        a.append(m_a.score(X_hold, y_hold))

        # (b) colorectal + stomach (transfer by augmentation)
        Xb = pd.concat([X_train, Xs], ignore_index=True)
        yb = np.concatenate([y_train, ys])
        m_b = RandomSurvivalForest(**RSF_PARAMS).fit(Xb, yb)
        b.append(m_b.score(X_hold, y_hold))

    stomach_only.append(float(np.mean(a)))
    with_transfer.append(float(np.mean(b)))
    print(f"  stomach n={n:4d}   stomach-only {np.mean(a):.3f}   +colorectal {np.mean(b):.3f}")

# ---- save numbers ----
pd.DataFrame({'stomach_train_n': sizes,
              'stomach_only': stomach_only,
              'with_colorectal_transfer': with_transfer}).to_csv(
    f'{RESULTS_DIR}/transfer_learning_curve.csv', index=False)
print(f"\nsizes         = {sizes}")
print(f"stomach_only  = {[round(v,3) for v in stomach_only]}")
print(f"with_transfer = {[round(v,3) for v in with_transfer]}")
print(f"pure_transfer = {round(pure_transfer,3)}")

In [24]:
# ---- plot (example style: transfer curve on top, from-scratch below) ----
fig, ax = plt.subplots(figsize=(7.6, 5))
ax.plot(sizes, with_transfer, '-o', color=RED,  lw=2.4, ms=6, label='Colorectal + stomach (transfer)')
ax.plot(sizes, stomach_only,  '-o', color=BLUE, lw=2.4, ms=6, label='Stomach only')
ax.axhline(pure_transfer, color=NAVY, ls='--', lw=1.5,
           label=f'Pure colorectal transfer ({pure_transfer:.3f})')
ax.axhline(0.5, color=GREY, ls=':', lw=1.2, label='Chance (0.5)')
ax.set_xlabel('Stomach patients used for training')
ax.set_ylabel('C-index on held-out stomach')
ax.set_title('Colorectal transfer helps most when stomach data is scarce', fontweight='bold')
ax.set_ylim(0.45, 0.72)
ax.legend(fontsize=9, frameon=False)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/fig_transfer_learning_curve.png', bbox_inches='tight', dpi=300)
plt.close()
print(f"\nSaved: {FIGURE_DIR}/fig_transfer_learning_curve.png")


Saved: Figures/fig_transfer_learning_curve.png
